In [1]:
import os
import re
import unicodedata
from collections import defaultdict, Counter

import pandas as pd
import numpy as np

BASE_DIR = ".."

TRAIN_DIR = os.path.join(BASE_DIR, "data", "train")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")

SOURCE1 = os.path.join(TRAIN_DIR, "train_source1.tsv")
SOURCE2 = os.path.join(TRAIN_DIR, "train_source2.tsv")
SOURCE3 = os.path.join(TRAIN_DIR, "train_source3.tsv")
GROUND_TRUTH = os.path.join(TRAIN_DIR, "train_ground_truth.tsv")

VALIDATION_IDS_FILE = os.path.join(
    OUTPUT_DIR,
    "validation_s1_ids.txt"
)

CHUNK_SIZE = 100_000

In [18]:
import pickle

BASELINE_PATH = os.path.join(
    OUTPUT_DIR,
    "blocking_experiment_01_baseline.pkl"
)

with open(BASELINE_PATH, "rb") as f:
    baseline_state = pickle.load(f)

exact_name_hits = baseline_state["exact_name_true_hits"]
exact_address_hits = baseline_state["exact_address_true_hits"]

exact_name_candidate_pairs = (
    baseline_state["exact_name_candidate_pairs"]
)

exact_address_candidate_pairs = (
    baseline_state["exact_address_candidate_pairs"]
)

total_true_matches = baseline_state["total_true_matches"]

print(
    "Exact-name true hits:",
    len(exact_name_hits)
)

print(
    "Exact-address true hits:",
    len(exact_address_hits)
)

print(
    "Exact-name candidate pairs:",
    exact_name_candidate_pairs
)

print(
    "Exact-address candidate pairs:",
    exact_address_candidate_pairs
)

Exact-name true hits: 167106
Exact-address true hits: 63073
Exact-name candidate pairs: 2171739
Exact-address candidate pairs: 76438


In [2]:
with open(VALIDATION_IDS_FILE, "r", encoding="utf-8") as f:
    validation_s1_ids = {
        line.strip()
        for line in f
        if line.strip()
    }

validation_s1_parts = []

for chunk in pd.read_csv(
    SOURCE1,
    sep="\t",
    dtype=str,
    chunksize=CHUNK_SIZE,
    keep_default_na=False
):
    selected = chunk[
        chunk["entity_id"].isin(validation_s1_ids)
    ]

    if len(selected) > 0:
        validation_s1_parts.append(selected)

validation_s1 = pd.concat(
    validation_s1_parts,
    ignore_index=True
)

validation_s1 = validation_s1.rename(
    columns={
        "entity_id": "s1_id",
        "business_name": "s1_name",
        "business_address": "s1_address",
        "country": "s1_country"
    }
)

print("Validation S1:", len(validation_s1))

Validation S1: 220682


In [3]:
def normalize_text(value):
    value = "" if value is None else str(value)

    value = unicodedata.normalize("NFKC", value)
    value = value.casefold()

    value = re.sub(
        r"[^\w\s]",
        " ",
        value,
        flags=re.UNICODE
    )

    value = re.sub(
        r"\s+",
        " ",
        value,
        flags=re.UNICODE
    )

    return value.strip()


def get_tokens(value):
    value = normalize_text(value)

    if not value:
        return set()

    return {
        token
        for token in value.split()
        if len(token) >= 2
    }

In [4]:
name_token_frequency = Counter()
address_token_frequency = Counter()

for row in validation_s1.itertuples(index=False):

    name_tokens = get_tokens(row.s1_name)
    address_tokens = get_tokens(row.s1_address)

    name_token_frequency.update(name_tokens)
    address_token_frequency.update(address_tokens)

print(
    "Unique name tokens:",
    len(name_token_frequency)
)

print(
    "Unique address tokens:",
    len(address_token_frequency)
)

Unique name tokens: 48652
Unique address tokens: 118619


In [5]:
print("Top name tokens:")
print(name_token_frequency.most_common(30))

print("\nTop address tokens:")
print(address_token_frequency.most_common(30))

Top name tokens:
[('limited', 52308), ('private', 43192), ('llc', 35516), ('inc', 23722), ('ltd', 14974), ('pvt', 12246), ('india', 5971), ('and', 5856), ('care', 5221), ('of', 4328), ('associates', 4277), ('group', 4009), ('llp', 3948), ('partners', 3561), ('center', 3455), ('corp', 3422), ('services', 3061), ('health', 2857), ('pc', 2759), ('co', 2755), ('clinic', 2716), ('solutions', 2627), ('global', 2528), ('trading', 2265), ('technologies', 2104), ('international', 2039), ('brothers', 2019), ('pllc', 2019), ('industries', 1913), ('medicine', 1846)]

Top address tokens:
[('road', 45852), ('no', 36025), ('street', 31080), ('drive', 22288), ('maharashtra', 19232), ('avenue', 19106), ('unit', 19020), ('floor', 16306), ('nagar', 15610), ('city', 14623), ('west', 13357), ('tx', 13179), ('delhi', 12180), ('pradesh', 11329), ('new', 11086), ('lane', 10699), ('ny', 10239), ('nc', 9475), ('mumbai', 8730), ('plot', 8620), ('oh', 8562), ('il', 7588), ('uttar', 7370), ('karnataka', 7005), ('t

In [6]:
name_freq_distribution = pd.Series(
    list(name_token_frequency.values())
).describe(
    percentiles=[
        0.50,
        0.75,
        0.90,
        0.95,
        0.99
    ]
)

address_freq_distribution = pd.Series(
    list(address_token_frequency.values())
).describe(
    percentiles=[
        0.50,
        0.75,
        0.90,
        0.95,
        0.99
    ]
)

print("NAME TOKEN FREQUENCY")
print(name_freq_distribution)

print("\nADDRESS TOKEN FREQUENCY")
print(address_freq_distribution)

NAME TOKEN FREQUENCY
count    48652.000000
mean        15.562402
std        386.824806
min          1.000000
50%          2.000000
75%          4.000000
90%          8.000000
95%         21.000000
99%        188.490000
max      52308.000000
dtype: float64

ADDRESS TOKEN FREQUENCY
count    118619.000000
mean         14.062191
std         284.773869
min           1.000000
50%           1.000000
75%           3.000000
90%          11.000000
95%          25.000000
99%         144.000000
max       45852.000000
dtype: float64


In [7]:
name_token_to_s1 = defaultdict(list)
address_token_to_s1 = defaultdict(list)

for row in validation_s1.itertuples(index=False):

    for token in get_tokens(row.s1_name):
        name_token_to_s1[token].append(
            row.s1_id
        )

    for token in get_tokens(row.s1_address):
        address_token_to_s1[token].append(
            row.s1_id
        )

print(
    "Name token index:",
    len(name_token_to_s1)
)

print(
    "Address token index:",
    len(address_token_to_s1)
)

Name token index: 48652
Address token index: 118619


In [8]:
validation_ground_truth = {}

for chunk in pd.read_csv(
    GROUND_TRUTH,
    sep="\t",
    dtype=str,
    chunksize=CHUNK_SIZE,
    keep_default_na=False
):
    for row in chunk.itertuples(index=False):

        s1_id = row.source1_entity_id

        if s1_id not in validation_s1_ids:
            continue

        value = row.matched_entity_ids.strip()

        if value:
            ids = {
                x.strip()
                for x in value.split(",")
                if x.strip()
            }
        else:
            ids = set()

        validation_ground_truth[s1_id] = ids

total_true_matches = sum(
    len(x)
    for x in validation_ground_truth.values()
)

true_target_to_s1 = {}

for s1_id, target_ids in validation_ground_truth.items():
    for target_id in target_ids:
        true_target_to_s1[target_id] = s1_id

print(
    "Validation true matches:",
    total_true_matches
)

Validation true matches: 764045


In [9]:
TOKEN_THRESHOLDS = [
    1,
    2,
    5,
    10,
    20,
    50
]

In [10]:
def evaluate_name_token_blocking(
    source_path,
    source_name
):
    candidate_counts = Counter()
    true_hits = {
        threshold: set()
        for threshold in TOKEN_THRESHOLDS
    }

    for chunk_number, chunk in enumerate(
        pd.read_csv(
            source_path,
            sep="\t",
            dtype=str,
            chunksize=CHUNK_SIZE,
            keep_default_na=False
        ),
        start=1
    ):

        for row in chunk.itertuples(index=False):

            target_id = row.entity_id
            country = row.country

            target_tokens = get_tokens(
                row.business_name
            )

            # Use a local set so the same S1 isn't counted
            # multiple times for several shared tokens.
            candidates_by_threshold = {
                threshold: set()
                for threshold in TOKEN_THRESHOLDS
            }

            for token in target_tokens:

                s1_list = name_token_to_s1.get(
                    token,
                    []
                )

                frequency = name_token_frequency.get(
                    token,
                    0
                )

                for threshold in TOKEN_THRESHOLDS:

                    if frequency <= threshold:

                        candidates_by_threshold[
                            threshold
                        ].update(s1_list)

            for threshold in TOKEN_THRESHOLDS:

                candidates = candidates_by_threshold[
                    threshold
                ]

                candidate_counts[
                    threshold
                ] += len(candidates)

                if target_id in true_target_to_s1:

                    true_s1 = true_target_to_s1[
                        target_id
                    ]

                    if true_s1 in candidates:

                        true_hits[
                            threshold
                        ].add(target_id)

        if chunk_number % 10 == 0:
            print(
                f"{source_name}: "
                f"{chunk_number * CHUNK_SIZE:,} rows processed"
            )

    return candidate_counts, true_hits

In [11]:
s2_name_counts, s2_name_hits = (
    evaluate_name_token_blocking(
        SOURCE2,
        "S2"
    )
)

s3_name_counts, s3_name_hits = (
    evaluate_name_token_blocking(
        SOURCE3,
        "S3"
    )
)

S2: 1,000,000 rows processed
S2: 2,000,000 rows processed
S2: 3,000,000 rows processed
S2: 4,000,000 rows processed
S2: 5,000,000 rows processed
S3: 1,000,000 rows processed
S3: 2,000,000 rows processed
S3: 3,000,000 rows processed
S3: 4,000,000 rows processed
S3: 5,000,000 rows processed


In [12]:
token_results = []

for threshold in TOKEN_THRESHOLDS:

    candidates = (
        s2_name_counts[threshold]
        +
        s3_name_counts[threshold]
    )

    hits = (
        len(s2_name_hits[threshold] | s3_name_hits[threshold])
    )

    token_results.append({
        "token_frequency_threshold": threshold,
        "candidate_pairs": candidates,
        "true_matches_captured": hits,
        "total_true_matches": total_true_matches,
        "candidate_recall": (
            hits / total_true_matches
        ),
        "avg_candidates_per_s1": (
            candidates
            / len(validation_s1_ids)
        )
    })

name_token_results_df = pd.DataFrame(
    token_results
)

name_token_results_df

,token_frequency_threshold,candidate_pairs,true_matches_captured,total_true_matches,candidate_recall,avg_candidates_per_s1
0,1,770438,66538,764045,0.087086,3.491168
1,2,2137307,113762,764045,0.148894,9.685008
2,5,7315868,204552,764045,0.267722,33.151177
3,10,15452201,271174,764045,0.354919,70.020215
4,20,32277845,314307,764045,0.411372,146.264059
5,50,83521249,362864,764045,0.474925,378.468788


In [17]:
union_results = []

for threshold in TOKEN_THRESHOLDS:

    token_hits = (
        s2_name_hits[threshold]
        |
        s3_name_hits[threshold]
    )

    new_hits = token_hits - exact_name_hits

    union_hits = token_hits | exact_name_hits

    union_results.append({
        "token_threshold": threshold,
        "exact_name_hits": len(exact_name_hits),
        "token_hits": len(token_hits),
        "new_hits_from_token_blocking": len(new_hits),
        "union_true_matches": len(union_hits),
        "union_recall": (
            len(union_hits)
            / total_true_matches
        )
    })

union_results_df = pd.DataFrame(
    union_results
)

union_results_df

,token_threshold,exact_name_hits,token_hits,new_hits_from_token_blocking,union_true_matches,union_recall
0,1,167106,66538,48840,215946,0.282635
1,2,167106,113762,83986,251092,0.328635
2,5,167106,204552,151236,318342,0.416653
3,10,167106,271174,200350,367456,0.480935
4,20,167106,314307,232783,399889,0.523384
5,50,167106,362864,267049,434155,0.568232


In [ ]:
address_union_results = []

for name_hits_label, base_hits in [
    ("name_exact", exact_name_hits)
]:

    address_hits = exact_address_hits

    new_from_address = (
        address_hits - base_hits
    )

    union_hits = (
        base_hits | address_hits
    )

    address_union_results.append({
        "base_route": name_hits_label,
        "base_true_matches": len(base_hits),
        "address_true_matches": len(address_hits),
        "new_matches_from_address": len(new_from_address),
        "union_true_matches": len(union_hits),
        "union_recall": (
            len(union_hits)
            / total_true_matches
        )
    })

address_union_df = pd.DataFrame(
    address_union_results
)

address_union_df

,base_route,base_true_matches,address_true_matches,new_matches_from_address,union_true_matches,union_recall
0,name_exact,167106,63073,53112,220218,0.288226


In [20]:
token_address_results = []

for threshold in TOKEN_THRESHOLDS:

    token_hits = (
        s2_name_hits[threshold]
        |
        s3_name_hits[threshold]
    )

    base_hits = exact_name_hits | token_hits

    new_from_address = (
        exact_address_hits - base_hits
    )

    final_union = (
        base_hits | exact_address_hits
    )

    token_address_results.append({
        "token_threshold": threshold,
        "name_token_true_matches": len(token_hits),
        "base_union_true_matches": len(base_hits),
        "new_matches_from_address": len(new_from_address),
        "final_union_true_matches": len(final_union),
        "final_union_recall": (
            len(final_union)
            / total_true_matches
        )
    })

token_address_union_df = pd.DataFrame(
    token_address_results
)

token_address_union_df

,token_threshold,name_token_true_matches,base_union_true_matches,new_matches_from_address,final_union_true_matches,final_union_recall
0,1,66538,215946,48850,264796,0.346571
1,2,113762,251092,45784,296876,0.388558
2,5,204552,318342,39964,358306,0.468959
3,10,271174,367456,35716,403172,0.527681
4,20,314307,399889,32966,432855,0.566531
5,50,362864,434155,29693,463848,0.607095


In [21]:
ADDRESS_TOKEN_THRESHOLDS = [1, 2, 5, 10]

In [22]:
address_token_frequency = Counter()

for row in validation_s1.itertuples(index=False):
    address_tokens = get_tokens(row.s1_address)
    address_token_frequency.update(address_tokens)

print("Unique address tokens:", len(address_token_frequency))
print(address_token_frequency.most_common(30))

Unique address tokens: 118619
[('road', 45852), ('no', 36025), ('street', 31080), ('drive', 22288), ('maharashtra', 19232), ('avenue', 19106), ('unit', 19020), ('floor', 16306), ('nagar', 15610), ('city', 14623), ('west', 13357), ('tx', 13179), ('delhi', 12180), ('pradesh', 11329), ('new', 11086), ('lane', 10699), ('ny', 10239), ('nc', 9475), ('mumbai', 8730), ('plot', 8620), ('oh', 8562), ('il', 7588), ('uttar', 7370), ('karnataka', 7005), ('tamil', 6401), ('nadu', 6399), ('tn', 6017), ('court', 6008), ('va', 5860), ('bangalore', 5858)]


In [23]:
address_token_to_s1 = defaultdict(list)

for row in validation_s1.itertuples(index=False):
    for token in get_tokens(row.s1_address):
        address_token_to_s1[token].append(row.s1_id)

print("Address token index:", len(address_token_to_s1))

Address token index: 118619


In [24]:
def evaluate_address_token_blocking(
    source_path,
    source_name
):
    candidate_counts = Counter()

    true_hits = {
        threshold: set()
        for threshold in ADDRESS_TOKEN_THRESHOLDS
    }

    for chunk_number, chunk in enumerate(
        pd.read_csv(
            source_path,
            sep="\t",
            dtype=str,
            chunksize=CHUNK_SIZE,
            keep_default_na=False
        ),
        start=1
    ):

        for row in chunk.itertuples(index=False):

            target_id = row.entity_id
            target_tokens = get_tokens(row.business_address)

            candidates_by_threshold = {
                threshold: set()
                for threshold in ADDRESS_TOKEN_THRESHOLDS
            }

            for token in target_tokens:

                frequency = address_token_frequency.get(
                    token,
                    0
                )

                if frequency == 0:
                    continue

                s1_list = address_token_to_s1.get(
                    token,
                    []
                )

                for threshold in ADDRESS_TOKEN_THRESHOLDS:

                    if frequency <= threshold:
                        candidates_by_threshold[
                            threshold
                        ].update(s1_list)

            for threshold in ADDRESS_TOKEN_THRESHOLDS:

                candidates = candidates_by_threshold[
                    threshold
                ]

                candidate_counts[
                    threshold
                ] += len(candidates)

                if target_id in true_target_to_s1:

                    true_s1 = true_target_to_s1[
                        target_id
                    ]

                    if true_s1 in candidates:
                        true_hits[
                            threshold
                        ].add(target_id)

        if chunk_number % 10 == 0:
            print(
                f"{source_name}: "
                f"{chunk_number * CHUNK_SIZE:,} rows processed"
            )

    return candidate_counts, true_hits

In [25]:
s2_address_counts, s2_address_hits = (
    evaluate_address_token_blocking(
        SOURCE2,
        "S2"
    )
)

s3_address_counts, s3_address_hits = (
    evaluate_address_token_blocking(
        SOURCE3,
        "S3"
    )
)

S2: 1,000,000 rows processed
S2: 2,000,000 rows processed
S2: 3,000,000 rows processed
S2: 4,000,000 rows processed
S2: 5,000,000 rows processed
S3: 1,000,000 rows processed
S3: 2,000,000 rows processed
S3: 3,000,000 rows processed
S3: 4,000,000 rows processed
S3: 5,000,000 rows processed


In [26]:
address_token_results = []

for threshold in ADDRESS_TOKEN_THRESHOLDS:

    hits = (
        s2_address_hits[threshold]
        |
        s3_address_hits[threshold]
    )

    candidates = (
        s2_address_counts[threshold]
        +
        s3_address_counts[threshold]
    )

    address_token_results.append({
        "address_token_threshold": threshold,
        "candidate_pairs": candidates,
        "true_matches_captured": len(hits),
        "total_true_matches": total_true_matches,
        "candidate_recall": (
            len(hits) / total_true_matches
        ),
        "avg_candidates_per_s1": (
            candidates / len(validation_s1_ids)
        )
    })

address_token_results_df = pd.DataFrame(
    address_token_results
)

address_token_results_df

,address_token_threshold,candidate_pairs,true_matches_captured,total_true_matches,candidate_recall,avg_candidates_per_s1
0,1,1375507,159397,764045,0.208623,6.232982
1,2,3538009,218864,764045,0.286454,16.032159
2,5,12210478,312005,764045,0.408359,55.330648
3,10,29528002,394062,764045,0.515758,133.803400


In [27]:
address_token_union_results = []

for threshold in ADDRESS_TOKEN_THRESHOLDS:

    address_token_hits = (
        s2_address_hits[threshold]
        |
        s3_address_hits[threshold]
    )

    # Name-token results for the corresponding threshold
    name_token_hits = (
        s2_name_hits[threshold]
        |
        s3_name_hits[threshold]
    )

    # Current name-based blocking
    name_base_hits = (
        exact_name_hits
        |
        name_token_hits
    )

    # Add address-token blocking
    combined_hits = (
        name_base_hits
        |
        address_token_hits
    )

    new_from_address_token = (
        address_token_hits
        - name_base_hits
    )

    address_overlaps_name = (
        address_token_hits
        &
        name_base_hits
    )

    address_token_union_results.append({
        "threshold": threshold,
        "name_base_hits": len(name_base_hits),
        "address_token_hits": len(address_token_hits),
        "new_matches_from_address_token": len(new_from_address_token),
        "overlap_with_name_base": len(address_overlaps_name),
        "combined_true_matches": len(combined_hits),
        "combined_recall": (
            len(combined_hits)
            / total_true_matches
        )
    })

address_token_union_df = pd.DataFrame(
    address_token_union_results
)

address_token_union_df

,threshold,name_base_hits,address_token_hits,new_matches_from_address_token,overlap_with_name_base,combined_true_matches,combined_recall
0,1,215946,159397,116342,43055,332288,0.434906
1,2,251092,218864,148661,70203,399753,0.523206
2,5,318342,312005,183123,128882,501465,0.656329
3,10,367456,394062,204675,189387,572131,0.748818


In [28]:
address_incremental_results = []

for threshold in ADDRESS_TOKEN_THRESHOLDS:

    address_token_hits = (
        s2_address_hits[threshold]
        |
        s3_address_hits[threshold]
    )

    new_over_exact_address = (
        address_token_hits
        - exact_address_hits
    )

    address_incremental_results.append({
        "threshold": threshold,
        "address_token_hits": len(address_token_hits),
        "exact_address_hits": len(exact_address_hits),
        "new_over_exact_address": len(new_over_exact_address),
        "address_token_overlap_exact_address": len(
            address_token_hits & exact_address_hits
        )
    })

address_incremental_df = pd.DataFrame(
    address_incremental_results
)

address_incremental_df

,threshold,address_token_hits,exact_address_hits,new_over_exact_address,address_token_overlap_exact_address
0,1,159397,63073,143165,16232
1,2,218864,63073,196511,22353
2,5,312005,63073,280272,31733
3,10,394062,63073,354374,39688


In [29]:
UNION_THRESHOLDS = [1, 2, 5, 10]

In [36]:
def evaluate_combined_candidate_volume(
    source_path,
    source_name,
    thresholds=UNION_THRESHOLDS
):
    total_candidate_pairs = {
        threshold: 0
        for threshold in thresholds
    }

    candidates_per_s1 = {
        threshold: Counter()
        for threshold in thresholds
    }

    total_rows = 0

    for chunk_number, chunk in enumerate(
        pd.read_csv(
            source_path,
            sep="\t",
            dtype=str,
            chunksize=CHUNK_SIZE,
            keep_default_na=False
        ),
        start=1
    ):

        total_rows += len(chunk)

        for row in chunk.itertuples(index=False):

            name_norm = normalize_text(row.business_name)
            address_norm = normalize_text(row.business_address)

            # Precompute candidate lists for exact name
            exact_name_candidates = set()

            if name_norm:
                exact_name_candidates.update(
                    name_exact_map.get(
                        name_norm,
                        []
                    )
                )

            # Precompute token lists
            target_name_tokens = get_tokens(
                row.business_name
            )

            target_address_tokens = get_tokens(
                row.business_address
            )

            for threshold in thresholds:

                combined_candidates = set(
                    exact_name_candidates
                )

                # Rare-name-token route
                for token in target_name_tokens:

                    frequency = name_token_frequency.get(
                        token,
                        0
                    )

                    if 0 < frequency <= threshold:

                        combined_candidates.update(
                            name_token_to_s1.get(
                                token,
                                []
                            )
                        )

                # Rare-address-token route
                for token in target_address_tokens:

                    frequency = address_token_frequency.get(
                        token,
                        0
                    )

                    if 0 < frequency <= threshold:

                        combined_candidates.update(
                            address_token_to_s1.get(
                                token,
                                []
                            )
                        )

                candidate_count = len(
                    combined_candidates
                )

                total_candidate_pairs[
                    threshold
                ] += candidate_count

                for s1_id in combined_candidates:

                    candidates_per_s1[
                        threshold
                    ][s1_id] += 1

        if chunk_number % 10 == 0:
            print(
                f"{source_name}: "
                f"{total_rows:,} rows processed"
            )

    return (
        total_candidate_pairs,
        candidates_per_s1
    )

In [40]:
from collections import defaultdict

# Make sure normalized columns exist in this notebook
if "name_norm" not in validation_s1.columns:
    validation_s1["name_norm"] = (
        validation_s1["s1_name"]
        .map(normalize_text)
    )

if "address_norm" not in validation_s1.columns:
    validation_s1["address_norm"] = (
        validation_s1["s1_address"]
        .map(normalize_text)
    )

# Build exact normalized-name index
name_exact_map = defaultdict(list)

for row in validation_s1.itertuples(index=False):
    if row.name_norm:
        name_exact_map[row.name_norm].append(row.s1_id)

print("Validation S1 rows:", len(validation_s1))
print("Unique exact-name keys:", len(name_exact_map))

Validation S1 rows: 220682
Unique exact-name keys: 187550


In [41]:
print(validation_s1.columns.tolist())

['s1_id', 's1_name', 's1_address', 's1_country', 'name_norm', 'address_norm']


In [42]:
s2_union_counts, s2_union_per_s1 = (
    evaluate_combined_candidate_volume(
        SOURCE2,
        "S2"
    )
)

S2: 1,000,000 rows processed
S2: 2,000,000 rows processed
S2: 3,000,000 rows processed
S2: 4,000,000 rows processed
S2: 5,000,000 rows processed


In [43]:
s3_union_counts, s3_union_per_s1 = (
    evaluate_combined_candidate_volume(
        SOURCE3,
        "S3"
    )
)

S3: 1,000,000 rows processed
S3: 2,000,000 rows processed
S3: 3,000,000 rows processed
S3: 4,000,000 rows processed
S3: 5,000,000 rows processed


In [44]:
union_volume_results = []

for threshold in UNION_THRESHOLDS:

    total_pairs = (
        s2_union_counts[threshold]
        +
        s3_union_counts[threshold]
    )

    combined_counter = (
        s2_union_per_s1[threshold] +
        s3_union_per_s1[threshold]
    )

    candidate_values = np.array(
        list(combined_counter.values()),
        dtype=np.int64
    )

    union_volume_results.append({
        "threshold": threshold,
        "candidate_pairs": total_pairs,
        "avg_candidates_per_s1": (
            total_pairs / len(validation_s1_ids)
        ),
        "median_candidates_per_s1": (
            np.median(candidate_values)
            if len(candidate_values)
            else 0
        ),
        "p95_candidates_per_s1": (
            np.percentile(candidate_values, 95)
            if len(candidate_values)
            else 0
        ),
        "max_candidates_per_s1": (
            np.max(candidate_values)
            if len(candidate_values)
            else 0
        )
    })

union_volume_df = pd.DataFrame(
    union_volume_results
)

union_volume_df

,threshold,candidate_pairs,avg_candidates_per_s1,median_candidates_per_s1,p95_candidates_per_s1,max_candidates_per_s1
0,1,4246987,19.244827,7.0,100.0,18970
1,2,7725301,35.006484,14.0,150.0,95240
2,5,21460174,97.244787,58.0,314.0,262826
3,10,46801923,212.078570,147.0,614.0,263003


In [45]:
combined_blocking_summary = (
    union_volume_df
    .merge(
        union_results_df[
            [
                "token_threshold",
                "union_true_matches",
                "union_recall"
            ]
        ],
        left_on="threshold",
        right_on="token_threshold",
        how="left"
    )
    .drop(
        columns=["token_threshold"]
    )
)

combined_blocking_summary

,threshold,candidate_pairs,avg_candidates_per_s1,median_candidates_per_s1,p95_candidates_per_s1,max_candidates_per_s1,union_true_matches,union_recall
0,1,4246987,19.244827,7.0,100.0,18970,215946,0.282635
1,2,7725301,35.006484,14.0,150.0,95240,251092,0.328635
2,5,21460174,97.244787,58.0,314.0,262826,318342,0.416653
3,10,46801923,212.078570,147.0,614.0,263003,367456,0.480935


In [46]:
correct_blocking_summary = (
    union_volume_df
    .merge(
        address_token_union_df[
            [
                "threshold",
                "combined_true_matches",
                "combined_recall"
            ]
        ],
        on="threshold",
        how="left"
    )
)

correct_blocking_summary

,threshold,candidate_pairs,avg_candidates_per_s1,median_candidates_per_s1,p95_candidates_per_s1,max_candidates_per_s1,combined_true_matches,combined_recall
0,1,4246987,19.244827,7.0,100.0,18970,332288,0.434906
1,2,7725301,35.006484,14.0,150.0,95240,399753,0.523206
2,5,21460174,97.244787,58.0,314.0,262826,501465,0.656329
3,10,46801923,212.078570,147.0,614.0,263003,572131,0.748818


In [47]:
correct_blocking_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "blocking_current_tradeoff.csv"
    ),
    index=False
)

In [48]:
import pickle
import os

CURRENT_THRESHOLD = 10

captured_true_targets = (
    exact_name_hits
    |
    s2_name_hits[CURRENT_THRESHOLD]
    |
    s3_name_hits[CURRENT_THRESHOLD]
    |
    s2_address_hits[CURRENT_THRESHOLD]
    |
    s3_address_hits[CURRENT_THRESHOLD]
)

captured_path = os.path.join(
    OUTPUT_DIR,
    "blocking_threshold_10_captured.pkl"
)

with open(captured_path, "wb") as f:
    pickle.dump(
        captured_true_targets,
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )

print("Saved:", captured_path)
print("Captured true targets:", len(captured_true_targets))
print(
    "Expected recall:",
    len(captured_true_targets) / total_true_matches
)

Saved: ../output/blocking_threshold_10_captured.pkl
Captured true targets: 572131
Expected recall: 0.7488184596457015
